# Interview Data Table Answers
Calculate answers for the data in the interview table using package functions. 

In [3]:
import numpy as np
import xarray as xr

from temp_humidity_index.calculate.calc_thi import _calculate_thi
from temp_humidity_index.plotting.plot_heat_risk_charts import categorise_thi

In [4]:
labels = ["Few", "Some", "Half", "Many", "All"]

# Build labelled dataset:
test_ds = xr.Dataset(
    data_vars={
        "dry_bulb_temp": (
            ["sample"],
            np.array([22.4, 18.6, 26.0, 13.2, 14.6, 34.2], dtype=float),
            {
                "units": "degC",
                "long_name": "Dry bulb temperature",
            }
        ),
        "wet_bulb_temp": (
            ["sample"],
            np.array([10.9, 16.44, 20.5, 9.4, 19.2, 22.8], dtype=float),
            {
                "units": "degC",
                "long_name": "Wet bulb temperature",
            }
        ),
    },
    coords={
        "sample": np.arange(6),
        "date": ("sample", [1, 1, 2, 2, 3, 3]),
        "location": ("sample", ["A", "B", "B", "C", "A", "C"]),
    },
)

# Calculate and add THI
thi = _calculate_thi(test_ds["dry_bulb_temp"], test_ds["wet_bulb_temp"])
test_ds["thi"] = thi

# Categorise THI and add category labels
test_ds["thi_index"] = categorise_thi(test_ds["thi"])
test_ds['category'] = xr.DataArray(
    data=[labels[int(i)] for i in test_ds["thi_index"].values],
    dims=["sample"],
)

display(test_ds)

<xarray.Dataset> Size: 408B
Dimensions:        (sample: 6)
Coordinates:
  * sample         (sample) int64 48B 0 1 2 3 4 5
    date           (sample) int64 48B 1 1 2 2 3 3
    location       (sample) <U1 24B 'A' 'B' 'B' 'C' 'A' 'C'
Data variables:
    dry_bulb_temp  (sample) float64 48B 22.4 18.6 26.0 13.2 14.6 34.2
    wet_bulb_temp  (sample) float64 48B 10.9 16.44 20.5 9.4 19.2 22.8
    thi            (sample) float64 48B 18.12 18.82 23.4 13.84 18.32 27.6
    thi_index      (sample) float64 48B 0.0 0.0 1.0 0.0 0.0 4.0
    category       (sample) <U4 96B 'Few' 'Few' 'Some' 'Few' 'Few' 'All'

In [5]:
df = test_ds.to_dataframe().reset_index()
df = df.drop(columns=["sample"])
df

,dry_bulb_temp,wet_bulb_temp,date,location,thi,thi_index,category
0,22.4,10.90,1,A,18.120,0.0,Few
1,18.6,16.44,1,B,18.816,0.0,Few
2,26.0,20.50,2,B,23.400,1.0,Some
3,13.2,9.40,2,C,13.840,0.0,Few
4,14.6,19.20,3,A,18.320,0.0,Few
5,34.2,22.80,3,C,27.600,4.0,All
